In [ ]:
!pip install geemap

In [ ]:
#Import libraries
import ee
import geemap

In [ ]:
import ee

# Step 1: Authenticate (opens a login window)
ee.Authenticate()

# Step 2: Initialize with your project
ee.Initialize(project='ee-arjunbirdi10')

print('GEE authenticated and ready!')

In [ ]:
# ─── CORRECTED GEE PYTHON CODE FOR GOOGLE COLAB ───
# The error: JavaScript uses {bands: [...]}
# Python requires  {'bands': [...]}  ← string keys!

import ee
import geemap

# Initialize (after authentication)
ee.Initialize(project='ee-arjunbirdi10')

# ── STUDY AREA ──
geometry = ee.Geometry.Rectangle([-125.5, 48.5, -123.0, 50.5])

# ── SENTINEL-2 COLLECTION ──
s2 = (ee.ImageCollection('COPERNICUS/S2_SR')
      .filterDate('2020-03-01', '2020-03-31')
      .filterBounds(geometry)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)))

# ── HSDI FUNCTION ──
def add_hsdi(image):
    hsdi = image.normalizedDifference(['B3', 'B4']).rename('HSDI')
    return image.addBands(hsdi)

s2_with_hsdi = s2.map(add_hsdi)

# ── MARCH 9 IMAGE ──
march9 = (s2_with_hsdi
          .filterDate('2020-03-09', '2020-03-10')
          .first())

# ── VISUALIZE WITH GEEMAP ──
Map = geemap.Map()
Map.centerObject(geometry, 9)

# True colour — NOTE: string keys in Python dicts!
Map.addLayer(march9, {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000
}, 'True Colour March 9 2020')

# HSDI layer
Map.addLayer(march9.select('HSDI'), {
    'min': -0.2667,
    'max': 0.7728,
    'palette': [
        '00008B', '0000FF', '00FFFF',
        '808080', 'FF4500', 'FF6347', 'FFFFFF'
    ]
}, 'HSDI Spawn Detection')

# Spawn mask
spawn_mask = march9.select('HSDI').gt(0.4757)
Map.addLayer(spawn_mask.selfMask(), {
    'palette': ['FF0000']
}, 'Spawn Detected (threshold > 0.4757)')

Map  # Display in Colab

In [ ]:
# ── LOAD YOUR ROIs FROM GEE ASSETS ──
# Replace with your actual asset paths
rois = ee.FeatureCollection('projects/ee-arjunbirdi10/assets/herring_spawn_ROIs')
print('ROI count:', rois.size().getInfo())
print('ROI properties:', rois.first().propertyNames().getInfo())

# ── LOAD YOUR SPECTRA CSV FROM GEE ASSETS ──
spectra = ee.FeatureCollection('projects/ee-arjunbirdi10/assets/saved_spectra-CSV')
print('Spectra rows:', spectra.size().getInfo())
print('Spectra columns:', spectra.first().propertyNames().getInfo())

In [ ]:
# Let's see the full column list and a sample row
print('All columns:', spectra.first().propertyNames().getInfo())
print('Sample row:', spectra.first().getInfo())
print('Unique classes:', spectra.aggregate_array('class').distinct().getInfo())

In [ ]:
# ── STEP 1: EXPORT SPECTRA TO GOOGLE DRIVE AS CSV ──

import pandas as pd
import ee

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Define the output path in Google Drive
output_file_path = 'herring_spawn_ml/spectra_training.csv'
drive_output_path = f'drive/MyDrive/{output_file_path}'

# Check if the file already exists in Drive to avoid re-exporting
import os
if os.path.exists(f'/content/{drive_output_path}'):
    print(f'File already exists at /content/{drive_output_path}. Skipping export.')
    df = pd.read_csv(f'/content/{drive_output_path}')
else:
    # Use ee.batch.Export.table.toDrive() for large datasets
    task = ee.batch.Export.table.toDrive(**{
        'collection': spectra,
        'description': 'Spectra_Export',
        'folder': 'herring_spawn_ml',
        'fileNamePrefix': 'spectra_training',
        'fileFormat': 'CSV'
    })
    task.start()

    print('Exporting spectra to Google Drive... Please wait for the task to complete.')
    print(f'Check task status: https://code.earthengine.google.com/tasks')

    # Wait for the task to complete (optional, can be done manually in EE Tasks tab)
    import time
    while task.active():
        print(f'Task is still active. Status: {task.status()}')
        time.sleep(10) # Wait for 10 seconds before checking again

    if task.status()['state'] == 'COMPLETED':
        print('Export completed successfully!')
        # Read the exported CSV from Google Drive
        df = pd.read_csv(f'/content/{drive_output_path}')
    else:
        raise Exception(f"Earth Engine export task failed with status: {task.status()['state']}")

# Clean up that BOM character in column name
df.columns = [c.replace('\ufeff', '') for c in df.columns]

print(df.head())
print(f"\nClass distribution:\n{df['class'].value_counts()}")
print(f"\nShape: {df.shape}")

print('Data loaded into DataFrame!')


In [ ]:
# ── STEP 1: EXPORT SPECTRA TO DRIVE (batch method) ──

task = ee.batch.Export.table.toDrive(
    collection=spectra,
    description='herring_spectra_export',
    folder='herring_spawn_ml',
    fileNamePrefix='spectra_training',
    fileFormat='CSV'
)
task.start()
print('Export started! Check task status below...')


In [ ]:
# ── CHECK EXPORT STATUS (re-run until it says COMPLETED) ──

print('Status:', task.status()['state'])


In [ ]:
# ── STEP 2: LOAD FROM DRIVE ──

from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/herring_spawn_ml/spectra_training.csv')
df.columns = [c.replace('\ufeff', '') for c in df.columns]

print(df.shape)
print(df['class'].value_counts())
print(df.head())


In [ ]:
# ── ML CLASSIFICATION ──

import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

# Features and target
feature_cols = ['B1', 'B2', 'B3', 'B4', 'B5']
X = df[feature_cols]
y = df['class']

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print('Classes:', le.classes_)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')

# Train models
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)

gb = GradientBoostingClassifier(n_estimators=200, random_state=42)
gb.fit(X_train, y_train)

print(f"\nRandom Forest Accuracy:     {rf.score(X_test, y_test):.4f}")
print(f"Gradient Boosting Accuracy: {gb.score(X_test, y_test):.4f}")

print('\n── Random Forest Report ──')
print(classification_report(y_test, rf.predict(X_test), target_names=le.classes_))

print('── Gradient Boosting Report ──')
print(classification_report(y_test, gb.predict(X_test), target_names=le.classes_))


In [ ]:
# ── CONFUSION MATRICES & FEATURE IMPORTANCE ──

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, model, name in [(axes[0], rf, 'Random Forest'), (axes[1], gb, 'Gradient Boosting')]:
    cm = confusion_matrix(y_test, model.predict(X_test))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
                xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
    ax.set_title(f'{name}\nAccuracy: {model.score(X_test, y_test):.4f}')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

# Feature importance
importances = pd.Series(rf.feature_importances_, index=feature_cols)
importances.sort_values().plot.barh(color='#2d6a4f', ax=axes[2])
axes[2].set_title('Feature Importance (RF)')
axes[2].set_xlabel('Importance')

plt.tight_layout()
plt.show()


In [ ]:
# ── DIAGNOSTIC: Band distributions + spatial clustering ──

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for i, band in enumerate(['B3', 'B2', 'B4']):
    for cls in ['spawn', 'substrate', 'water']:
        subset = df[df['class'] == cls]
        axes[i].hist(subset[band], bins=40, alpha=0.5, label=cls, density=True)
    axes[i].set_title(f'{band} Distribution by Class')
    axes[i].set_xlabel('Reflectance')
    axes[i].legend()

plt.tight_layout()
plt.show()

# Spatial spread of each class
print('\n── Spatial spread per class ──')
for cls in ['spawn', 'substrate', 'water']:
    subset = df[df['class'] == cls]
    print(f'{cls:12s} | Samples: {len(subset):5d} | '
          f'Lat: {subset["Latitude"].min():.3f}–{subset["Latitude"].max():.3f} | '
          f'Lon: {subset["Longitude"].min():.3f}–{subset["Longitude"].max():.3f}')

In [ ]:
# ── SPATIAL CROSS-VALIDATION ──
# Group pixels by approximate polygon using spatial clustering

from sklearn.cluster import KMeans
from sklearn.model_selection import LeaveOneGroupOut

# Cluster pixels into ~5 spatial groups per class (approximating your polygons)
df['spatial_group'] = -1

for cls in ['spawn', 'substrate', 'water']:
    mask = df['class'] == cls
    coords = df.loc[mask, ['Latitude', 'Longitude']]
    km = KMeans(n_clusters=5, random_state=42, n_init=10)
    df.loc[mask, 'spatial_group'] = km.fit_predict(coords) + (
        {'spawn': 0, 'substrate': 10, 'water': 20}[cls]
    )

print('Spatial groups per class:')
print(df.groupby('class')['spatial_group'].nunique())

# Leave-One-Group-Out CV — each fold holds out one polygon
X = df[['B1', 'B2', 'B3', 'B4', 'B5']]
y_encoded = le.transform(df['class'])
groups = df['spatial_group']

logo = LeaveOneGroupOut()
rf = RandomForestClassifier(n_estimators=200, random_state=42)

fold_scores = []
for fold, (train_idx, test_idx) in enumerate(logo.split(X, y_encoded, groups)):
    rf.fit(X.iloc[train_idx], y_encoded[train_idx])
    score = rf.score(X.iloc[test_idx], y_encoded[test_idx])
    fold_scores.append(score)

print(f'\n── Spatial CV Results ──')
print(f'Per-fold accuracy: {[f"{s:.4f}" for s in fold_scores]}')
print(f'Mean accuracy:     {np.mean(fold_scores):.4f}')
print(f'Std deviation:     {np.std(fold_scores):.4f}')
print(f'\nThis is your REAL expected accuracy on unseen locations.')


In [ ]:
# ── SPATIAL CROSS-VALIDATION ──
# Group pixels by approximate polygon using spatial clustering

from sklearn.cluster import KMeans
from sklearn.model_selection import LeaveOneGroupOut

# Cluster pixels into ~5 spatial groups per class (approximating your polygons)
df['spatial_group'] = -1

for cls in ['spawn', 'substrate', 'water']:
    mask = df['class'] == cls
    coords = df.loc[mask, ['Latitude', 'Longitude']]
    km = KMeans(n_clusters=5, random_state=42, n_init=10)
    df.loc[mask, 'spatial_group'] = km.fit_predict(coords) + (
        {'spawn': 0, 'substrate': 10, 'water': 20}[cls]
    )

print('Spatial groups per class:')
print(df.groupby('class')['spatial_group'].nunique())

# Leave-One-Group-Out CV — each fold holds out one polygon
X = df[['B1', 'B2', 'B3', 'B4', 'B5']]
y_encoded = le.transform(df['class'])
groups = df['spatial_group']

logo = LeaveOneGroupOut()
rf = RandomForestClassifier(n_estimators=200, random_state=42)

fold_scores = []
for fold, (train_idx, test_idx) in enumerate(logo.split(X, y_encoded, groups)):
    rf.fit(X.iloc[train_idx], y_encoded[train_idx])
    score = rf.score(X.iloc[test_idx], y_encoded[test_idx])
    fold_scores.append(score)

print(f'\n── Spatial CV Results ──')
print(f'Per-fold accuracy: {[f"{s:.4f}" for s in fold_scores]}')
print(f'Mean accuracy:     {np.mean(fold_scores):.4f}')
print(f'Std deviation:     {np.std(fold_scores):.4f}')
print(f'\nThis is your REAL expected accuracy on unseen locations.')


In [ ]:
# ── IDENTIFY WHICH POLYGONS ARE HARD & WHY ──

from sklearn.metrics import confusion_matrix

logo = LeaveOneGroupOut()
rf = RandomForestClassifier(n_estimators=200, random_state=42)

print('── Problem Folds Breakdown ──\n')
for fold, (train_idx, test_idx) in enumerate(logo.split(X, y_encoded, groups)):
    rf.fit(X.iloc[train_idx], y_encoded[train_idx])
    score = rf.score(X.iloc[test_idx], y_encoded[test_idx])

    if score < 0.95:  # Only show problem folds
        preds = rf.predict(X.iloc[test_idx])
        group_id = df.iloc[test_idx]['spatial_group'].iloc[0]
        cls_name = df.iloc[test_idx]['class'].iloc[0]

        print(f'Fold {fold} | Group {group_id} | Class: {cls_name} | '
              f'Accuracy: {score:.4f} | Samples: {len(test_idx)}')

        cm = confusion_matrix(y_encoded[test_idx], preds, labels=[0,1,2])
        print(f'  Confusion [spawn, substrate, water]:')
        for i, row_label in enumerate(le.classes_):
            print(f'    {row_label:12s} → predicted as: '
                  f'spawn={cm[i][0]}, substrate={cm[i][1]}, water={cm[i][2]}')
        print()

# ── ADD HSDI AS A FEATURE TO IMPROVE ACCURACY ──

df['HSDI'] = (df['B3'] - df['B4']) / (df['B3'] + df['B4'])

feature_cols_v2 = ['B1', 'B2', 'B3', 'B4', 'B5', 'HSDI']
X_v2 = df[feature_cols_v2]

rf2 = RandomForestClassifier(n_estimators=200, random_state=42)

fold_scores_v2 = []
for train_idx, test_idx in logo.split(X_v2, y_encoded, groups):
    rf2.fit(X_v2.iloc[train_idx], y_encoded[train_idx])
    fold_scores_v2.append(rf2.score(X_v2.iloc[test_idx], y_encoded[test_idx]))

print(f'── With HSDI Added ──')
print(f'Per-fold accuracy: {[f"{s:.4f}" for s in fold_scores_v2]}')
print(f'Mean accuracy:     {np.mean(fold_scores_v2):.4f} (was {np.mean(fold_scores):.4f})')
print(f'Std deviation:     {np.std(fold_scores_v2):.4f} (was {np.std(fold_scores):.4f})')

In [ ]:
# ── FINAL MODEL: Train on all data with HSDI ──

from sklearn.ensemble import RandomForestClassifier
import joblib

feature_cols_final = ['B1', 'B2', 'B3', 'B4', 'B5', 'HSDI']

X_final = df[feature_cols_final]
y_final = le.transform(df['class'])

rf_final = RandomForestClassifier(n_estimators=300, random_state=42)
rf_final.fit(X_final, y_final)

# Save model
joblib.dump(rf_final, '/content/drive/MyDrive/herring_spawn_ml/rf_spawn_model.joblib')
joblib.dump(le, '/content/drive/MyDrive/herring_spawn_ml/label_encoder.joblib')
print('Model saved to Drive!')

# ── FEATURE IMPORTANCE (final model) ──
import matplotlib.pyplot as plt
import pandas as pd

importances = pd.Series(rf_final.feature_importances_, index=feature_cols_final)
importances.sort_values().plot.barh(color='#2d6a4f', figsize=(8, 4))
plt.title('Final Model — Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

# ── SUMMARY TABLE FOR THESIS ──
print('\n══════════════════════════════════════════')
print('  HERRING SPAWN ML CLASSIFICATION SUMMARY')
print('══════════════════════════════════════════')
print(f'  Training samples:    {len(df)}')
print(f'  Classes:             spawn, substrate, water')
print(f'  Features:            B1–B5 + HSDI')
print(f'  Model:               Random Forest (300 trees)')
print(f'  Spatial CV Accuracy: {np.mean(fold_scores_v2):.4f} ± {np.std(fold_scores_v2):.4f}')
print(f'  Key confusion:       substrate ↔ spawn (ecologically expected)')
print(f'  Top feature:         B3 (Green band)')
print(f'  HSDI impact:         +3.9% accuracy, −4.7% std dev')
print('══════════════════════════════════════════')


In [ ]:
# ── FINAL MODEL: Train on all data with HSDI ──

from sklearn.ensemble import RandomForestClassifier
import joblib

feature_cols_final = ['B1', 'B2', 'B3', 'B4', 'B5', 'HSDI']

X_final = df[feature_cols_final]
y_final = le.transform(df['class'])

rf_final = RandomForestClassifier(n_estimators=300, random_state=42)
rf_final.fit(X_final, y_final)

# Save model
joblib.dump(rf_final, '/content/drive/MyDrive/herring_spawn_ml/rf_spawn_model.joblib')
joblib.dump(le, '/content/drive/MyDrive/herring_spawn_ml/label_encoder.joblib')
print('Model saved to Drive!')

# ── FEATURE IMPORTANCE (final model) ──
import matplotlib.pyplot as plt
import pandas as pd

importances = pd.Series(rf_final.feature_importances_, index=feature_cols_final)
importances.sort_values().plot.barh(color='#2d6a4f', figsize=(8, 4))
plt.title('Final Model — Feature Importance')
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

# ── SUMMARY TABLE FOR THESIS ──
print('\n══════════════════════════════════════════')
print('  HERRING SPAWN ML CLASSIFICATION SUMMARY')
print('══════════════════════════════════════════')
print(f'  Training samples:    {len(df)}')
print(f'  Classes:             spawn, substrate, water')
print(f'  Features:            B1–B5 + HSDI')
print(f'  Model:               Random Forest (300 trees)')
print(f'  Spatial CV Accuracy: {np.mean(fold_scores_v2):.4f} ± {np.std(fold_scores_v2):.4f}')
print(f'  Key confusion:       substrate ↔ spawn (ecologically expected)')
print(f'  Top feature:         B3 (Green band)')
print(f'  HSDI impact:         +3.9% accuracy, −4.7% std dev')
print('══════════════════════════════════════════')


In [ ]:
# ── APPLY TRAINED RF TO FULL SENTINEL-2 IMAGE IN GEE ──
# This runs the classification on the full March 9 image

import ee
import geemap
import numpy as np

# Use your trained model's rules to build a GEE classifier
# First, train an equivalent ee.Classifier on the same data

# Load your ROIs
rois = ee.FeatureCollection('projects/ee-arjunbirdi10/assets/herring_spawn_ROIs')

# Function to add a geometry to each feature
def add_geometry(feature):
  return feature.setGeometry(
      ee.Geometry.Point([feature.get('Longitude'), feature.get('Latitude')])
  )

# Map the function over the collection to add geometries
rois_with_geometry = rois.map(add_geometry)

# Load March 9 image with HSDI
geometry = ee.Geometry.Rectangle([-125.5, 48.5, -123.0, 50.5])

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterDate('2020-03-09', '2020-03-10')
      .filterBounds(geometry)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
      .first())

# Add HSDI band
hsdi = s2.normalizedDifference(['B3', 'B4']).rename('HSDI')
image = s2.addBands(hsdi)

# Select classification bands (matching your Colab model)
bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'HSDI']

# Encode class labels as integers for GEE classifier
# spawn=0, substrate=1, water=2
class_lookup = ee.Dictionary({
    'spawn': 0,
    'substrate': 1,
    'water': 2
})

def encode_class(feature):
    label = feature.get('class')
    return feature.set('class_int', class_lookup.get(label))

rois_encoded = rois_with_geometry.map(encode_class)

# Sample the image at ROI locations
training = image.select(bands).sampleRegions(
    collection=rois_encoded,
    properties=['class_int'],
    scale=10
)

# Train GEE Random Forest (mirrors your Colab model)
classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=300
).train(
    features=training,
    classProperty='class_int',
    inputProperties=bands
)

# Classify the full image
classified = image.select(bands).classify(classifier)

print('Classification complete!')
print('Classes: 0=spawn, 1=substrate, 2=water')


In [ ]:
# ── DEBUG: Check ROI asset structure ──

rois = ee.FeatureCollection('projects/ee-arjunbirdi10/assets/herring_spawn_ROIs')

print('ROI count:', rois.size().getInfo())
print('First ROI properties:', rois.first().getInfo()['properties'])
print('Geometry type:', rois.first().geometry().type().getInfo())

# Check if ROIs overlap with the image
print('\nImage date:', image.date().format('YYYY-MM-dd').getInfo())
print('Image bounds:', image.geometry().bounds().getInfo())

# Test: sample just one pixel
test_sample = image.select(bands).sampleRegions(
    collection=rois.limit(5),
    properties=['class'],
    scale=10
)
print('Test sample size:', test_sample.size().getInfo())


In [ ]:
# ── FIX: Rebuild geometries from Lat/Lon columns ──

def add_geometry(feature):
    lat = feature.get('Latitude')
    lon = feature.get('Longitude')
    point = ee.Geometry.Point([lon, lat])
    return feature.setGeometry(point)

rois_fixed = rois.map(add_geometry)

# Verify it worked
print('Geometry type:', rois_fixed.first().geometry().type().getInfo())
print('Sample coords:', rois_fixed.first().geometry().coordinates().getInfo())

# Now re-encode classes and sample
class_lookup = ee.Dictionary({
    'spawn': 0,
    'substrate': 1,
    'water': 2
})

def encode_class(feature):
    label = feature.get('class')
    return feature.set('class_int', class_lookup.get(label))

rois_encoded = rois_fixed.map(encode_class)

# Sample image at fixed ROI locations
bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'HSDI']

training = image.select(bands).sampleRegions(
    collection=rois_encoded,
    properties=['class_int'],
    scale=10
)
print('Training samples:', training.size().getInfo())

# Train classifier
classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=300
).train(
    features=training,
    classProperty='class_int',
    inputProperties=bands
)

# Classify
classified = image.select(bands).classify(classifier)
spawn_mask = classified.eq(0)

print('Classification complete!')


In [ ]:
# ── DISPLAY CLASSIFIED MAP ──

Map = geemap.Map()
Map.centerObject(geometry, 9)

# True colour base
Map.addLayer(image, {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000
}, 'True Colour', False)

# HSDI layer
Map.addLayer(image.select('HSDI'), {
    'min': -0.27,
    'max': 0.77,
    'palette': ['00008B', '0000FF', '00FFFF', '808080',
                'FF4500', 'FF6347', 'FFFFFF']
}, 'HSDI', False)

# ML Classification result
Map.addLayer(classified, {
    'min': 0,
    'max': 2,
    'palette': [
        'FF0000',  # 0 = spawn (red)
        'FFA500',  # 1 = substrate (orange)
        '0000FF',  # 2 = water (blue)
    ]
}, 'ML Classification')

# Spawn-only mask
spawn_mask = classified.eq(0)
Map.addLayer(spawn_mask.selfMask(), {
    'palette': ['FF0000']
}, 'Spawn Only (ML)')

# Compare with your original HSDI threshold
hsdi_spawn = image.select('HSDI').gt(0.4757)
Map.addLayer(hsdi_spawn.selfMask(), {
    'palette': ['FFFF00']
}, 'Spawn (HSDI Threshold)', False)

# Add legend
legend_dict = {
    'Spawn (ML)': 'FF0000',
    'Substrate': 'FFA500',
    'Water': '0000FF',
    'Spawn (HSDI threshold)': 'FFFF00'
}
Map.add_legend(title='Classification', legend_dict=legend_dict)

Map


In [ ]:
# ── FIX: Mask land before classifying ──

# Use a water mask (multiple options, JRC Global Surface Water is reliable)
water_occurrence = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence')
water_mask = water_occurrence.gt(50)  # pixels that are water >50% of the time

# Apply mask to image BEFORE classifying
image_water = image.updateMask(water_mask)

# Re-add HSDI on masked image
hsdi = image_water.normalizedDifference(['B3', 'B4']).rename('HSDI')
image_water = image_water.addBands(hsdi)

# Classify only water pixels
classified = image_water.select(bands).classify(classifier)
spawn_mask = classified.eq(0)

# Re-display
Map = geemap.Map()
Map.centerObject(geometry, 9)

Map.addLayer(image, {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0, 'max': 3000
}, 'True Colour', False)

Map.addLayer(classified, {
    'min': 0, 'max': 2,
    'palette': ['FF0000', 'FFA500', '0000FF']
}, 'ML Classification (water only)')

Map.addLayer(spawn_mask.selfMask(), {
    'palette': ['FF0000']
}, 'Spawn Only (ML)')

# HSDI threshold comparison
hsdi_spawn = image_water.select('HSDI').gt(0.4757)
Map.addLayer(hsdi_spawn.selfMask(), {
    'palette': ['FFFF00']
}, 'Spawn (HSDI Threshold)', False)

legend_dict = {
    'Spawn (ML)': 'FF0000',
    'Substrate': 'FFA500',
    'Water': '0000FF',
    'Spawn (HSDI threshold)': 'FFFF00'
}
Map.add_legend(title='Classification', legend_dict=legend_dict)

Map



In [ ]:
# ── ACCURACY ASSESSMENT IN GEE ──

# Confusion matrix from GEE classifier
conf_matrix = classifier.confusionMatrix()
print('GEE Confusion Matrix:')
print(conf_matrix.getInfo())
print(f'Overall Accuracy: {conf_matrix.accuracy().getInfo():.4f}')
print(f'Kappa: {conf_matrix.kappa().getInfo():.4f}')

# Producers & consumers accuracy
print(f"Producer's Accuracy: {conf_matrix.producersAccuracy().getInfo()}")
print(f"Consumer's Accuracy: {conf_matrix.consumersAccuracy().getInfo()}")


In [ ]:
# ── CALCULATE SPAWN AREA ──

# ML-detected spawn area
spawn_area_ml = spawn_mask.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=geometry,
    scale=10,
    maxPixels=1e9
)

# HSDI threshold spawn area
spawn_area_hsdi = hsdi_spawn.multiply(ee.Image.pixelArea()).reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=geometry,
    scale=10,
    maxPixels=1e9
)

ml_hectares = ee.Number(spawn_area_ml.get('classification')).divide(10000)
hsdi_hectares = ee.Number(spawn_area_hsdi.get('HSDI')).divide(10000)

print(f'ML Spawn Area:   {ml_hectares.getInfo():.2f} hectares')
print(f'HSDI Spawn Area: {hsdi_hectares.getInfo():.2f} hectares')
print(f'Difference:      {abs(ml_hectares.getInfo() - hsdi_hectares.getInfo()):.2f} hectares')


In [ ]:
# ── TIME SERIES: Server-side computation + export ──

def classify_image(img):
    hsdi = img.normalizedDifference(['B3', 'B4']).rename('HSDI')
    img_with_hsdi = img.addBands(hsdi)

    # Apply water mask
    water_occurrence = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').select('occurrence')
    water_mask = water_occurrence.gt(50)  # pixels that are water >50% of the time
    img_masked = img_with_hsdi.updateMask(water_mask)

    # Classify
    cls = img_masked.select(bands).classify(classifier)
    spawn_px = cls.eq(0)

    # Calculate spawn area
    area = spawn_px.multiply(ee.Image.pixelArea()).reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=geometry,
        scale=10,
        maxPixels=1e9
    )

    return ee.Feature(None, {
        'date': img.date().format('YYYY-MM-dd'),
        'spawn_area_m2': area.get('classification')
    })

# Apply to March 2020 collection
s2_march = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
            .filterDate('2020-03-01', '2020-03-31')
            .filterBounds(geometry)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10)))

print('Images in March 2020:', s2_march.size().getInfo())

# Convert to FeatureCollection and export
ts_fc = ee.FeatureCollection(s2_march.map(classify_image))

task = ee.batch.Export.table.toDrive(
    collection=ts_fc,
    description='spawn_timeseries_march2020',
    folder='herring_spawn_ml',
    fileNamePrefix='spawn_timeseries',
    fileFormat='CSV'
)
task.start()
print('Time series export started!')

In [ ]:
# ── CHECK STATUS (re-run until COMPLETED) ──
print('Status:', task.status()['state'])


In [ ]:
# ── LOAD AND PLOT TIME SERIES ──

import pandas as pd
import matplotlib.pyplot as plt

ts = pd.read_csv('/content/drive/MyDrive/herring_spawn_ml/spawn_timeseries.csv')
ts['date'] = pd.to_datetime(ts['date'])
ts['spawn_hectares'] = ts['spawn_area_m2'] / 10000
ts = ts.sort_values('date')

print(ts[['date', 'spawn_hectares']])

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(ts['date'], ts['spawn_hectares'], 'o-', color='#d62828',
        linewidth=2, markersize=8)
ax.fill_between(ts['date'], ts['spawn_hectares'], alpha=0.15, color='#d62828')
ax.set_title('Herring Spawn Extent — March 2020 (ML Classification)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Spawn Area (hectares)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── CLEAN TIME SERIES: Average per date + flag outliers ──

# Group by date and average (handles overlapping orbits)
ts_clean = ts.groupby('date')['spawn_hectares'].mean().reset_index()
ts_clean = ts_clean.sort_values('date')

print(ts_clean.to_string(index=False))

# Plot cleaned version
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(ts_clean['date'], ts_clean['spawn_hectares'],
       width=0.6, color='#d62828', alpha=0.7, edgecolor='#8b0000')
ax.set_title('Herring Spawn Extent — March 2020\n(ML Classification, daily average)', fontsize=14)
ax.set_xlabel('Date')
ax.set_ylabel('Spawn Area (hectares)')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Summary stats
print(f'\nPeak spawn date:  {ts_clean.loc[ts_clean["spawn_hectares"].idxmax(), "date"]}')
print(f'Peak spawn area:  {ts_clean["spawn_hectares"].max():,.0f} hectares')
print(f'Mean spawn area:  {ts_clean["spawn_hectares"].mean():,.0f} hectares')
print(f'Total image dates: {len(ts_clean)}')


In [ ]:
# ── EXPORT CLASSIFIED MAP TO DRIVE ──

# Export classified image as GeoTIFF
export_task = ee.batch.Export.image.toDrive(
    image=classified.toByte(),
    description='herring_spawn_classified_march9_2020',
    folder='herring_spawn_ml',
    region=geometry,
    scale=10,
    maxPixels=1e9,
    fileFormat='GeoTIFF'
)
export_task.start()
print('Classified map export started!')

# Export spawn-only mask
spawn_export = ee.batch.Export.image.toDrive(
    image=spawn_mask.selfMask().toByte(),
    description='herring_spawn_mask_march9_2020',
    folder='herring_spawn_ml',
    region=geometry,
    scale=10,
    maxPixels=1e9,
    fileFormat='GeoTIFF'
)
spawn_export.start()
print('Spawn mask export started!')

# Export time series CSV
ts.to_csv('/content/drive/MyDrive/herring_spawn_ml/spawn_timeseries_march2020.csv',
          index=False)
print('Time series CSV saved!')

print('\nAll exports running — check Drive in a few minutes.')
